# 🎯 [K12_M05] 모르는 질문 담당 부서로 안내하기
### 실습 솔루션 & 정답 해설 (Google Colab / Jupyter)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/edunlab/milicontent2/blob/main/uploads/K12/k12_m05_solution.ipynb)

---

## 📌 미션 개요

| 항목 | 내용 |
| :--- | :--- |
| **과정 / 과제** | 일반행정·부대관리 (Level 1) / K12 행정 업무 FAQ 챗봇 설계 |
| **미션** | 미션 05. 모르는 질문 담당 부서로 안내하기 (소요시간: 약 1.5시간) |
| **실습 유형** | 생성형 AI 시스템 프롬프트 엔지니어링 및 팩트체크 검증 워크북 |

> **⚠️ [교육용 가상 자료 원칙]**
> 본 실습의 모든 자료는 안전 규정에 따라 **교육용 가상 부대 자료(가상 한빛대대)**로 구성되어 있습니다.
> 실제 군사 기밀이나 부대 세부 인적·작전 정보는 외부 AI 도구에 절대 입력하지 않습니다.



In [ ]:
# ============================================================================
# [환경 설정 및 헬퍼 함수] 마크다운 시각화 렌더러 및 가상 데이터셋 로더
# ============================================================================
from IPython.display import display, Markdown
import json
import os

def load_dataset(mission_num):
    """미션 번호(1~10)에 해당하는 가상 데이터셋 JSON을 로드하여 반환합니다."""
    filename = f"k12_m{mission_num:02d}_dataset.json"
    for base in [".", "uploads/K12", "../uploads/K12"]:
        filepath = os.path.join(base, filename)
        if os.path.exists(filepath):
            with open(filepath, "r", encoding="utf-8") as f:
                return json.load(f)
    print(f"⚠️ 데이터셋 파일을 찾을 수 없습니다: {filename}")
    return None

def show_dataset_summary(mission_num):
    """해당 미션의 데이터셋 주요 항목을 요약 출력합니다."""
    data = load_dataset(mission_num)
    if data:
        display(Markdown(f"**[데이터셋 로드 성공]** `{data.get('title', '')}` ({data.get('mission', '')})"))
        keys = [k for k in data.keys() if k not in ["project", "mission", "title"]]
        print(f"포함 항목: {', '.join(keys)}")

def show_result(mission_title, prompt, result):
    """학습자가 입력한 프롬프트와 AI 답변 및 검증 결과를 시각적으로 렌더링하는 헬퍼 함수"""
    p_text = prompt.strip() if prompt.strip() else "(작성된 프롬프트가 없습니다)"
    r_text = result.strip() if result.strip() else "(기록된 결과가 없습니다)"
    
    md_content = f"""
---
### 📌 [{mission_title}] 실습 기록 리포트

#### 💬 입력 프롬프트
```text
{p_text}
```

#### 📋 AI 생성 결과 및 팩트체크 검증 내역
{r_text}
---
"""
    display(Markdown(md_content))

print("✅ 워크북 시각화 헬퍼 함수 및 가상 데이터셋 로더(k12_m01~m10) 준비 완료!")



---

## 🎯 미션 05. 모르는 질문 담당 부서로 안내하기
**소요시간**: 약 1.5시간

![미션 5 FAQ 범위 밖 예외 질의 차단 및 담당 부서 연계 표준 Fallback 가드레일 다이어그램](/uploads/common/K12_m05_safety_fallback.png)

### 📋 미션 개요 및 배경
제공된 FAQ에 수록되지 않은 질문이 유입되었을 때 AI가 근거 없는 답변을 창작(할루시네이션)하는 현상을 통제하는 예외 처리(Fallback) 방침을 수립합니다. '휴가 일수', '증명서 수수료' 등 데이터 범위를 벗어난 질문을 질의하여 취약점을 점검하고, 정보 부재 시 "FAQ에 명시되지 않음"을 분명히 알린 후 적절한 담당 부서(행정과, 인사과, 당직실 등)로 안내하도록 규칙을 보강합니다. AI 시스템의 안전 경계를 설정하고 신뢰도를 보장하는 방어적 프롬프팅 기법을 체득합니다.

### 🎯 미션 학습목표
'휴가 일수는 며칠인가요', '증명서 수수료는 얼마인가요' 같은 자료 밖 질문 4개를 넣어 봅니다. 'FAQ에 없음'이라고 알리고 가장 가까운 담당 부서(행정과, 인사과, 중대 행정반, 당직실)를 안내하는 규칙을 추가해 지어낸 답이 사라졌는지 확인합니다. **이를 통해 데이터 범위 밖의 불확실한 질의에 대한 허위 생성을 차단하고, 안전하게 담당 부서로 연결하는 예외 처리(Fallback) 및 거절 프롬프팅 기술을 익힙니다.**

### 📑 실습 가이드 및 세부 단계 (프롬프트 & 검증 포인트)
#### 🔹 Step 1. 질문 카드 17~20을 넣는다.
#### 이 단계의 역할

제공된 FAQ에 수록되지 않은 4개 질문('휴가 일수는 며칠인가요', '증명서 수수료는 얼마인가요' 등)을 질의하여 챗봇의 취약점을 노출시킵니다.

#### 해야 할 일

- FAQ에 없는 질문 카드 17~20번(가상 한빛대대 FAQ 범위를 벗어난 질의)을 확인합니다.

- 현재 챗봇 세션에 4개 질문을 순차적으로 질의하고 응답을 기록합니다.

- 챗봇이 "자료에 없다"고 하는지, 아니면 인터넷 상식을 기반으로 허위 일수/규정을 지어내는지(할루시네이션) 관찰합니다.

#### 💡 AI 프롬프트 질의 예시

> <p>**[자료 밖 질문 스트레스 테스트]:** "{현재 시스템 프롬프트 및 FAQ}

[장병 질문 17]: 육군 복무 중 정기휴가 연가 총일수는 며칠이고 포상휴가는 최대 몇 일까지 저축해서 쓸 수 있습니까?"</p>

#### 🔍 검증 포인트 (원문 대조 및 팩트체크)

- **할루시네이션 포착:** FAQ에 없는 일수나 규정을 마치 한빛대대 공식 규정인 것처럼 그럴듯하게 답하는지 확인
- **위험성 평가:** 잘못된 행정 정보 전달로 인한 장병 피해 가능성을 진단

#### 핵심 확인 지표

> 자료 밖 질문 4종 취약점 테스트 로그

#### 학습자가 제출해야 하는 내용

- 자료 밖 질문 응답 기록표

#### 🔹 Step 2. FAQ에 없으면 안내 문구와 담당 부서를 답하게 규칙을 넣는다.
#### 이 단계의 역할

FAQ에 없는 내용에 대해 "FAQ에 수록되지 않은 내용입니다"를 명시하고 가장 인접한 담당 부서(행정과, 인사과, 중대 행정반, 당직실)와 내선 번호를 안내하는 Fallback 지시문을 프롬프트에 추가합니다.

#### 해야 할 일

- 프롬프트에 "제공된 FAQ 표에 없는 내용은 절대 추측하여 답하지 말고, '해당 문의는 FAQ에 수록되어 있지 않습니다'라고 고지하라"는 규칙을 추가합니다.

- 문의 유형별 담당 부서 매칭 안내문(복무일수→인사과, 비용→행정과, 당일 긴급→당직실)을 예외 규칙으로 작성합니다.

- 프롬프트에 예외 처리(Fallback) 블록을 명확히 삽입합니다.

#### 💡 AI 프롬프트 질의 예시

> <p>**[예외 처리 Fallback 프롬프트 규칙 작성]:** "AI가 FAQ 표에서 질문에 대한 답을 찾을 수 없는 경우, 추측하여 답하는 것을 전면 금지하고 다음과 같이 응답하도록 시스템 프롬프트 규칙을 보강해 줘:
1) '해당 문의는 행정 업무 FAQ에 수록되어 있지 않습니다.' 명시
2) 문의 유형에 따라 가장 적절한 부서(인사과, 행정과, 중대 행정반, 당직실) 안내"</p>

#### 🔍 검증 포인트 (원문 대조 및 팩트체크)

- **거절의 명확성:** '모릅니다'로 끝나는 것이 아니라 공식 부서로 안전하게 인계하는가?
- **추측 발언 차단:** 단 한 줄의 임의 추측도 섞이지 않았는가?

#### 핵심 확인 지표

> Fallback 지침이 포함된 개선 시스템 프롬프트

#### 학습자가 제출해야 하는 내용

- 예외 처리 규칙 추가 프롬프트 전문

#### 🔹 Step 3. 전후를 비교한다.
#### 이 단계의 역할

규칙 추가 전의 할루시네이션 응답과 규칙 추가 후의 안전한 거절/부서 안내 응답을 1:1 비교표로 작성합니다.

#### 해야 할 일

- 예외 처리 규칙이 반영된 프롬프트에 질문 17~20번을 다시 질의합니다.

- 규칙 적용 전(지어냄)과 적용 후(안내 불가 고지 및 담당 부서 안내)의 응답을 나란히 비교표로 정리합니다.

- AI의 거짓 답변(할루시네이션)이 0건으로 통제되었는지 최종 판정합니다.

#### 💡 AI 프롬프트 질의 예시

> **[Fallback 적용 전후 비교표 작성 질의]:** "자료 밖 질문 4개에 대해 예외 처리 규칙 적용 전(허위 답변/추측)과 적용 후(안내 불가 고지 및 담당 부서 안내)의 응답 차이와 보안 안전성 향상 효과를 표로 정리해 줘."

#### 🔍 검증 포인트 (원문 대조 및 팩트체크)

- **환각 완전 제거:** 4개 질문 모두에서 추측 답변이 완전히 사라졌는가?
- **부서 안내 정확성:** 휴가 일수는 인사과, 시설 파손은 군수과/당직실 등 올바른 부서로 인계되었는가?

#### 핵심 확인 지표

> 자료 밖 질문 4종 전후 비교 분석표

#### 학습자가 제출해야 하는 내용

- 예외 처리 전후 비교표

#### 🔹 Step 4. 심화 도전 과제 및 최적화
#### 이 단계의 역할

야간이나 주말에 긴급 행정 문의가 발생했을 때 당직실로 인계하는 시간대별 예외 처리 규칙을 추가합니다.

#### 해야 할 일

- 일과 후(17:30 이후) 또는 휴일 발생 문의에 대한 안내 지침을 구상합니다.

- "긴급 문의는 당직사령실(내선 0000)로 문의하십시오"라는 시간대별 예외 분기 규칙을 추가합니다.

#### 💡 AI 프롬프트 질의 예시

> **[시간대별 긴급 Fallback 지침 질의]:** "야간이나 주말에 발생하는 긴급 행정 질의에 대해 당직실로 즉시 연결하도록 유도하는 시간대 분기 프롬프트 지시문을 작성해 줘."

#### 🔍 검증 포인트 (원문 대조 및 팩트체크)

- **긴급 대응 적합성:** 부대 비상 소통 체계와 모순되지 않는가?

#### 핵심 확인 지표

> 시간대별 긴급 대응 Fallback 규칙 메모

#### 학습자가 제출해야 하는 내용

- 심화 도전 과제 메모

### 📦 미션 필수 학습자료 및 팁
**미션 수행 필수 학습자료**

- **FAQ 범위 밖 예외 질문 카드 4종 (질문 17~20):** 총 휴가 일수, 증명서 수수료, 복무기간 단축 등 FAQ 데이터셋에 수록되지 않은 함정 질의 세트
- **안전 Fallback(예외 처리) 규칙 프롬프트 템플릿:** 정보 부재 시 "FAQ에 수록되지 않음" 고지 및 인사과·행정과·당직실 공식 부서 안내문 서식
- **[학습 가이드] 안전 프롬프팅 및 Fallback(거절) 메커니즘:** AI가 모르는 질문에 대해 지어내지 않고 정해진 안전 규정에 따라 담당 부서로 연결하는 방어적 지시문 설계법

**실습 팁:** 거절 지침을 줄 때는 단순히 "모르면 답하지 마"라고 하면 AI가 당황하거나 사회 일반론을 답하므로, "FAQ에 명시되지 않은 질문은 '해당 문의는 행정 FAQ에 없습니다'를 명시하고 관련 부서로 안내하라"며 정형화된 거절 문구를 직접 지정해 주어야 합니다.

### 🔍 평가 기준 (PASS 조건)
#### 미션 판정

**평가 구분:** 필수

**핵심 평가 기준:** [필수] 예외 처리·환각 방지 — FAQ 범위 밖 질문에 대해 거짓 답변을 차단하고, 올바른 담당 부서로 안내하는 Fallback 규칙을 구현했는가? (PASS / FAIL)

#### 미션 PASS 조건

- Fallback 예외 처리 지시문 템플릿 작성 완료

- 자료 밖 질문 4종에 대한 임의 추측(환각) 0건 달성

- 질문 성격별 올바른 부서 및 내선 번호 안내 전후 비교표 완성

#### 채점자 판정 메모

> Q17(연가 총일수) 질의 시 AI가 숫자를 지어내지 않고 단호하게 "행정 FAQ에 없습니다"라고 선을 긋는지 확인한다.



In [ ]:
# [미션 05.] 정답 해설 - 모범 프롬프트 및 완성 산출물 리포트
mission_05_prompt = """
[FAQ 외 질문 안전 Fallback(예외 대응) 및 부서 연계 규칙 수립]
너는 AI 챗봇 환각 방어 및 예외 처리 전문가야. 장병들이 제공된 FAQ에 없는 질문(예: 주말 병영식당 메뉴, 이번 달 급여일, 유격 훈련 일정 등)을 물었을 때, AI가 임의로 거짓 답변을 지어내지 않고 단호하고 친절하게 안내를 거절하는 '안전 Fallback 지시문'을 작성해 줘.
- 필수 규칙:
1. "문의하신 내용은 행정 FAQ에 등록되어 있지 않습니다" 고지
2. 장병이 혼선을 겪지 않도록 실제 담당 부서(인사과, 군수과, 당직실 등) 및 내선 번호 안내
3. 예외 질의 4건에 대해 Fallback 적용 전(환각 위험)과 적용 후(안전 거절 및 부서 연계)의 응답 비교표를 작성해 줘.
"""

mission_05_result = """
## 1. 안전 Fallback(예외 대응) 지시문 템플릿

```markdown
[예외 처리 및 Fallback 규칙]
1. 사용자의 질문이 [참조 행정 FAQ 데이터]에 명시되어 있지 않은 경우, 절대로 자체 지식으로 추측하여 답변하지 마십시오.
2. FAQ에 없는 질문에 대해서는 반드시 다음 2단계 형식으로만 답변하십시오:
   - 1) "문의하신 내용은 부대 행정 업무 FAQ에 수록되어 있지 않습니다."
   - 2) 질문 내용과 가장 연관된 담당 부서 및 내선 번호를 안내하고 공식 지휘계통에 확인할 것을 권고하십시오.
      * 복무 일수, 계급, 징계, 포상 규정 문의 → 대대 인사과 (내선 102)
      * 행정 제반 서류, 일반 문의 → 중대 행정반 (내선 101)
      * 병영 시설, 비품, 피복 보급 문의 → 대대 군수과 (내선 106)
      * 전산, 인트라넷, 보안 문의 → 대대 통신소대 (내선 108)
      * 야간/휴일 긴급 문의 → 대대 당직사령실 (내선 100)
```

---

## 2. 예외 처리 전후 응답 비교표

| 질문 | 규칙 적용 전 (환각 발생) | 규칙 적용 후 (안전 Fallback) | 판정 및 효과 |
| :--- | :--- | :--- | :--- |
| **Q1. 정기휴가 연가 총일수** | "육군 일반 병사의 복무 중 정기휴가는 총 24일입니다." *(FAQ에 없는데 임의 추측)* | "문의하신 연가 총일수 규정은 부대 행정 업무 FAQ에 수록되어 있지 않습니다. 정확한 본인 연가 잔여일수는 **대대 인사과(내선 102)** 또는 중대 행정반으로 문의하시기 바랍니다." | **환각 차단 완료**<br>(공식 부서 연계) |
| **Q2. 증명서 발급 수수료** | "국방부 증명서는 전부 무료이며 10원도 내지 않습니다." *(FAQ에 근거 없음)* | "문의하신 제증명서 발급 수수료 관련 내용은 행정 FAQ에 명시되어 있지 않습니다. **대대 인사과(내선 102)**에 확인해 주시기 바랍니다." | **환각 차단 완료** |
| **Q3. 복무기간 단축 계산** | "2018년 이후 입대자는 최대 90일까지 단축되며..." *(인터넷 법령 검색 혼용)* | "개인별 복무기간 산정은 행정 FAQ에 포함되어 있지 않습니다. 병적 기록 조회가 필요한 사안이므로 **대대 인사과(내선 102)**로 문의하십시오." | **오안내 방지 완료** |
| **Q4. 민간 병원 외출 기준** | "민간 병원은 원칙적으로 주말에만 외출이 가능합니다." *(잘못된 규정 왜곡)* | "민간 의료기관 사적 진료 절차는 행정 FAQ에 수록되어 있지 않습니다. **대대 의무실(내선 107)** 군의관 상담 및 행정반 승인을 거치시기 바랍니다." | **안전 경계 수립 완료** |
"""

# 렌더러 실행 (모범 해설 결과를 시각적으로 렌더링합니다)
show_result("미션 05. 모르는 질문 담당 부서로 안내하기", mission_05_prompt, mission_05_result)

